# Просмотр фиксированных результатов

Численные первоисточники — JSON завершённого эксперимента. Рост модельной награды не является измеренным эффектом для пользователей. Ноутбук не переобучает модели и не выбирает параметры.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
if not (ROOT / 'results').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'results/research_v2/summary.json').exists(), 'Откройте ноутбук из каталога проекта или notebooks/'
summary = json.loads((ROOT / 'results/research_v2/summary.json').read_text())
results = json.loads((ROOT / 'results/research_v2/evaluations/holdout/results.json').read_text())


## Финальная когорта

Новые пользователи того же периода; это не независимая проверка переноса во времени.

In [ ]:
summary['holdout']['summary']


## Основные парные сравнения

Средние по трём сохранённым seed-моделям. Интервалы условны на этих весах и временном периоде.

In [ ]:
for name in ['ratio', 'time']:
    values = summary['seed_results'][name]
    print(name, 'ΔNDCG@10:', values['comparisons']['ndcg@10'])
    reward = 'policy_expected_ratio' if name == 'ratio' else 'policy_expected_time_reward'
    print('Δreward:', values['comparisons'][reward])


## Сравнение политик

NDCG и Recall относятся к следующему логированному объекту; награда — прогноз замороженной reward model.

In [ ]:
names = ['baseline_s11', 'rl_ratio_b0.1_h1_s11', 'rl_time_b1_h1_s11', 'sft_s11']
for name in names:
    m = results['models'][name]
    print(name, {key: m[key] for key in ['ndcg@10', 'recall@10', 'policy_expected_ratio', 'policy_expected_duration_limited_seconds']})


## Сохранённые иллюстрации

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for axis, name in zip(axes, ['calibration', 'tradeoff', 'rollout']):
    axis.imshow(mpimg.imread(ROOT / 'reports/figures' / (name + '.png')))
    axis.set_title(name)
    axis.axis('off')
plt.tight_layout()
